# Feature Distribution Analysis

Exploratory analysis of engineered ride-sharing features for the Roadies-CityRide project.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

from roadies.ingestion.loaders import load_csv
from roadies.features.demand_supply import engineer_demand_supply_features
from roadies.features.surge import engineer_surge_features
from roadies.features.acceptance import engineer_acceptance_features
from roadies.features.cancellation import engineer_cancellation_features
from roadies.features.experience import engineer_experience_features
from roadies.features.demand_period import classify_high_demand
from roadies.analysis.distributions import (
    compute_numerical_stats,
    compute_categorical_stats,
    compare_high_demand,
    compare_cities,
)

In [ ]:
# Load and engineer features
df = load_csv(Path('data/raw/rides.csv'))
df, _ = engineer_demand_supply_features(df)
df, _ = engineer_surge_features(df)
df, _ = engineer_acceptance_features(df)
df, _ = engineer_cancellation_features(df)
df, _ = engineer_experience_features(df)
df, _ = classify_high_demand(df)
print(f'Dataset: {len(df)} rows, {len(df.columns)} columns')

## Numerical Distributions

In [ ]:
numerical_metrics = [
    'city_hour_requested_rides', 'city_hour_available_drivers',
    'demand_supply_ratio', 'surge_multiplier', 'surge_intensity',
    'driver_acceptance_rate', 'wait_time_minutes', 'trip_duration_minutes',
    'trip_distance_km', 'base_fare',
]
stats = compute_numerical_stats(df, numerical_metrics)
pd.DataFrame([vars(s) for s in stats]).round(2)

## Surge Distribution

In [ ]:
fig = px.histogram(df, x='surge_multiplier', nbins=30, title='Surge Multiplier Distribution')
fig.update_layout(xaxis_title='Surge Multiplier', yaxis_title='Count')
fig.show()

## City-Level Comparison

In [ ]:
city_stats = df.groupby('city').agg(
    rides=('ride_id', 'count'),
    avg_surge=('surge_multiplier', 'mean'),
    avg_wait=('wait_time_minutes', 'mean'),
    acceptance_rate=('was_accepted', 'mean'),
    cancellation_rate=('rider_cancelled', 'mean'),
).round(3)
city_stats

In [ ]:
fig = px.bar(city_stats.reset_index(), x='city', y='rides', title='Ride Volume by City')
fig.show()

## High-Demand vs Normal Comparison

In [ ]:
hd_comparison = compare_high_demand(df, [
    'surge_multiplier', 'wait_time_minutes',
    'driver_acceptance_rate', 'city_hour_available_drivers',
])
pd.DataFrame([vars(c) for c in hd_comparison]).round(3)